# Capstone Projects

Every lesson so far handed you an architecture and asked you to fill in the pieces.
This one does not. Four projects, each combining several lessons, each with a
specification and a bar to clear — and no solution notebook telling you what the
code should look like.

That is the point. The gap between "I completed the tutorial" and "I can build
this" is exactly the gap this lesson exists to close.

**How to use this notebook**

- Pick **one** project and finish it properly. A finished translator teaches more
  than four half-built prototypes.
- Each project lists a **minimum viable version** and a set of **extensions**. Get
  the minimum working end to end before touching an extension.
- The scaffolding below is real, runnable, and shared: data loading, a training
  harness, evaluation metrics, and a checklist. The architecture is yours.
- Every project fits on an M4 MacBook Air with 24 GB, using only the dependencies
  already in this repository.

Read [Chapter 12 of the book](../book/index.html#ch12) for guidance on scoping,
and revisit [Chapter 08](../book/index.html#ch08) when something inevitably breaks.

## Shared Setup

In [1]:
import copy
import math
import random
import time
from collections import Counter

import nltk
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available()
                      else 'cpu')

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print('torch  :', torch.__version__)
print('device :', device)
print()
print('Corpora available without any extra download beyond NLTK:')
for pkg in ('gutenberg', 'movie_reviews', 'comtrans', 'punkt'):
    nltk.download(pkg, quiet=True)
from nltk.corpus import gutenberg, movie_reviews, comtrans
print(f'  gutenberg     : {len(gutenberg.fileids())} texts, e.g. {gutenberg.fileids()[:3]}')
print(f'  movie_reviews : {len(movie_reviews.fileids())} labelled reviews')
print(f'  comtrans      : {comtrans.fileids()}')

torch  : 2.12.0
device : mps

Corpora available without any extra download beyond NLTK:
  gutenberg     : 18 texts, e.g. ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt']
  movie_reviews : 2000 labelled reviews
  comtrans      : ['alignment-de-en.txt', 'alignment-de-fr.txt', 'alignment-en-fr.txt']


---
## The Harness

Reusable pieces so you spend your time on the interesting part. These are the
patterns from [lesson 08](../08_pytorch_in_practice/solution.ipynb), collected.

In [2]:
class Vocab:
    """Word-level vocabulary with the four special tokens used throughout."""

    PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'
    PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

    def __init__(self, sentences, max_size=8000, min_freq=1):
        self.itos = [self.PAD, self.SOS, self.EOS, self.UNK]
        freq = Counter(tok for s in sentences for tok in s)
        for tok, c in freq.most_common():
            if len(self.itos) >= max_size:
                break
            if c >= min_freq:
                self.itos.append(tok)
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens, max_len=None, add_special=True):
        ids = [self.stoi.get(t, self.UNK_IDX) for t in tokens]
        if max_len is not None:
            ids = ids[:max_len]
        if add_special:
            ids = [self.SOS_IDX] + ids + [self.EOS_IDX]
        return ids

    def decode(self, ids, strip_special=True):
        toks = [self.itos[i] for i in ids]
        if strip_special:
            toks = [t for t in toks if t not in (self.PAD, self.SOS, self.EOS)]
        return ' '.join(toks)

    def pad_to(self, ids, length):
        return ids + [self.PAD_IDX] * (length - len(ids))


def train_model(model, train_loader, val_loader, criterion, epochs=20, lr=1e-3,
                clip=1.0, warmup_frac=0.05, patience=None, verbose=True):
    """Generic training loop: warmup + cosine, gradient clipping, best-checkpoint restore.

    Expects each batch to be (inputs, targets) and the model to map inputs -> logits
    that `criterion` can consume directly. Override `step_fn` for anything more
    exotic (teacher forcing, for instance).
    """
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = epochs * len(train_loader)
    warm = max(1, int(warmup_frac * total_steps))
    step = 0

    hist = {'train': [], 'val': []}
    best_val, best_state, best_epoch, since_improved = float('inf'), None, 0, 0

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            lr_now = lr * (step + 1) / warm if step < warm else \
                0.1 * lr + 0.9 * lr * 0.5 * (1 + math.cos(
                    math.pi * (step - warm) / max(1, total_steps - warm)))
            for g in opt.param_groups:
                g['lr'] = lr_now

            loss = criterion(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            running += loss.item() * xb.size(0)
            step += 1

        tr = running / len(train_loader.dataset)

        model.eval()
        vtotal = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vtotal += criterion(model(xb), yb).item() * xb.size(0)
        va = vtotal / len(val_loader.dataset)

        hist['train'].append(tr); hist['val'].append(va)

        if va < best_val:
            best_val, best_epoch, since_improved = va, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            since_improved += 1

        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch == 1):
            print(f'  epoch {epoch:3d} | train {tr:.4f} | val {va:.4f} | lr {lr_now:.2e}')

        if patience and since_improved >= patience:
            print(f'  early stop at epoch {epoch} (no improvement for {patience})')
            break

    model.load_state_dict(best_state)
    print(f'  restored epoch {best_epoch} (val {best_val:.4f})')
    return hist


def plot_history(hist, title='Training'):
    plt.figure(figsize=(7, 3))
    plt.plot(hist['train'], label='train', color='#6c5ce7', lw=2)
    plt.plot(hist['val'], label='val', color='#e17055', lw=2, ls='--')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.title(title)
    plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


def count_params(model):
    total = sum(p.numel() for p in set(model.parameters()))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


print('harness ready')

harness ready


In [3]:
def bleu(hypothesis, reference, max_n=4):
    """Sentence-level BLEU with a brevity penalty. Both args are token lists.

    Not a substitute for reading outputs yourself, but it makes runs comparable.
    """
    if not hypothesis:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        h = Counter(tuple(hypothesis[i:i + n]) for i in range(len(hypothesis) - n + 1))
        r = Counter(tuple(reference[i:i + n]) for i in range(len(reference) - n + 1))
        overlap = sum(min(c, r[g]) for g, c in h.items())
        total = max(1, sum(h.values()))
        # smoothing so a single missing n-gram order does not zero the whole score
        precisions.append((overlap + 1e-9) / total)
    geo = math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if len(hypothesis) > len(reference) else \
        math.exp(1 - len(reference) / max(1, len(hypothesis)))
    return bp * geo


def corpus_bleu(hyps, refs):
    return float(np.mean([bleu(h, r) for h, r in zip(hyps, refs)]))


def perplexity(loss_nats):
    return math.exp(loss_nats)


def distinct_n(tokens, n=2):
    """Fraction of distinct n-grams — a cheap repetition detector for generators."""
    if len(tokens) < n:
        return 0.0
    grams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    return len(set(grams)) / len(grams)


# quick self-check
ref = 'the cat sat on the mat'.split()
print(f'BLEU identical      : {bleu(ref, ref):.3f}')
print(f'BLEU close          : {bleu("the cat sat on a mat".split(), ref):.3f}')
print(f'BLEU unrelated      : {bleu("dogs run quickly today here now".split(), ref):.3f}')
print(f'distinct-2 varied   : {distinct_n("a b c d e f".split()):.2f}')
print(f'distinct-2 repeating: {distinct_n("a b a b a b".split()):.2f}')

BLEU identical      : 1.000
BLEU close          : 0.537
BLEU unrelated      : 0.000
distinct-2 varied   : 1.00
distinct-2 repeating: 0.40


---
# Project 1 — A Translator

**Builds on:** lessons 06, 07, 09
**Difficulty:** moderate · **Time:** 3–5 hours

Build an English→French translator that beats the lesson 09 baseline, and be able
to say *why* it is better with a number rather than a vibe.

### Minimum viable version

1. Load `comtrans` `alignment-en-fr.txt` — and name the file explicitly, or you
   will train on three language pairs at once (see lesson 09).
2. Build separate source and target vocabularies from the **training split only**.
3. Implement an encoder–decoder transformer with cross-attention and causal masking.
4. Train with teacher forcing and `ignore_index=PAD_IDX`.
5. Implement greedy decoding.
6. Report validation loss, corpus BLEU on a held-out set, and print 10 translations
   with their references.

**Bar to clear:** BLEU above the greedy baseline you measure first, and at least
five of ten sample translations recognisably conveying the source meaning.

### Extensions, roughly in order of payoff

- **Beam search** (width 3–5). Usually worth 1–3 BLEU over greedy. Remember to
  normalise by length or the model will prefer short outputs.
- **Subword tokenisation.** Word-level vocabularies produce endless `<UNK>`s on
  French morphology. A simple BPE over the training text will cut them sharply.
- **Length bucketing** (lesson 08) to cut padding waste; measure the speedup.
- **Label smoothing** (`nn.CrossEntropyLoss(label_smoothing=0.1)`) — standard in
  translation, worth about a point of BLEU.
- **Attention visualisation** for a sentence where the word order differs between
  languages. If the model has learned alignment, you will see it.
- **Both directions** — train fr→en with the same code and compare. Which is
  harder, and does your explanation match the data?

### Where it will go wrong

Overfitting, quickly: the filtered corpus is under 10k pairs. Use best-checkpoint
restore from the start. And if outputs loop ("de la session de la session…"), that
is exposure bias, not a bug — read the lesson 09 discussion before trying to fix it.

In [4]:
# ── Project 1 starter: data only. The model is yours. ────────────────────────────
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
MAX_LEN = 16

pairs = comtrans.aligned_sents('alignment-en-fr.txt')   # NAME THE FILE

en_sents, fr_sents = [], []
for p in pairs:
    src = [w.lower() for w in p.words]
    tgt = [w.lower() for w in p.mots]
    if 1 <= len(src) <= MAX_LEN and 1 <= len(tgt) <= MAX_LEN:
        en_sents.append(src)
        fr_sents.append(tgt)

# Split BEFORE building vocabularies, so the test set cannot leak into them.
idx = list(range(len(en_sents)))
random.Random(SEED).shuffle(idx)
cut = int(0.9 * len(idx))
train_idx, val_idx = idx[:cut], idx[cut:]

en_vocab = Vocab([en_sents[i] for i in train_idx], max_size=4000)
fr_vocab = Vocab([fr_sents[i] for i in train_idx], max_size=4000)

def make_tensors(indices):
    L = MAX_LEN + 2
    src = torch.tensor([en_vocab.pad_to(en_vocab.encode(en_sents[i], MAX_LEN), L)
                        for i in indices], dtype=torch.long)
    tgt = torch.tensor([fr_vocab.pad_to(fr_vocab.encode(fr_sents[i], MAX_LEN), L)
                        for i in indices], dtype=torch.long)
    return src, tgt

src_tr, tgt_tr = make_tensors(train_idx)
src_va, tgt_va = make_tensors(val_idx)

print(f'pairs      : {len(en_sents):,}  ({len(train_idx):,} train / {len(val_idx):,} val)')
print(f'vocab      : en {len(en_vocab):,}  fr {len(fr_vocab):,}')
print(f'tensors    : {tuple(src_tr.shape)}')
print()
unk_rate = (src_tr == UNK_IDX).float().mean().item()
print(f'<UNK> rate on the English side: {100*unk_rate:.1f}%')
print('  (if this is high, that is your first extension: subword tokenisation)')
print()
for i in range(3):
    print(f'  EN: {en_vocab.decode(src_tr[i].tolist())}')
    print(f'  FR: {fr_vocab.decode(tgt_tr[i].tolist())}')
    print()

# YOUR CODE: build the Seq2SeqTransformer, train it, decode, score with corpus_bleu.

pairs      : 9,606  (8,645 train / 961 val)
vocab      : en 4,000  fr 4,000
tensors    : (8645, 18)

<UNK> rate on the English side: 1.8%
  (if this is high, that is your first extension: subword tokenisation)

  EN: lisbon european council
  FR: conseil européen de lisbonne

  EN: i want to concentrate on kosovo .
  FR: pour ma part , je <UNK> mon propos sur la situation au kosovo .

  EN: in this context , the e-europe initiative comes at just the right moment .
  FR: dans ce contexte , l ' initiative e-europe vient à point <UNK> .



---
# Project 2 — A Text Generator

**Builds on:** lessons 04, 10
**Difficulty:** moderate · **Time:** 3–5 hours

Train a language model on an author of your choice and generate text that is
recognisably in their style. Then evaluate it honestly, which is harder than
training it.

### Minimum viable version

1. Pick a corpus from `gutenberg` — Austen, Melville, Shakespeare, the KJV Bible.
   Or concatenate several and see what happens.
2. Decide character-level or word-level, and be able to justify the choice.
3. Build a decoder-only transformer (lesson 10) with causal masking.
4. Train with warmup + cosine decay to a validation perplexity you record.
5. Implement temperature, top-k, and nucleus sampling.
6. Generate 500 tokens at four sampling settings and compare them side by side.

**Bar to clear:** validation perplexity well below the uniform baseline
(`vocab_size`), and samples that hold spelling and punctuation together for at
least a full sentence.

### Extensions

- **A KV cache** (lesson 10) and a measured speedup on a long generation.
- **Repetition penalty** — divide the logits of already-generated tokens by a
  factor before sampling. Measure `distinct_n` before and after.
- **Scaling study.** Train at 2, 4, and 8 layers. Plot final loss against parameter
  count and against wall-clock time. Which one is the honest x-axis?
- **Compare against lesson 04's RNN** on the identical corpus at a matched parameter
  count. The comparison is only meaningful if the counts match — see the book's
  exercise 05.2.
- **Quantize it** (lesson 11) and measure how far you can push bit-width before the
  samples visibly degrade. Do it blind, ideally.
- **Prompt continuation.** Feed the opening line of a different book and see how
  quickly the model reverts to its training style.

### Evaluating a generator honestly

Perplexity measures next-token prediction, not quality — a model can have excellent
perplexity and generate tedious text. Use several signals: perplexity,
`distinct_n` for repetition, and your own reading. Reading twenty samples is not
optional.

In [5]:
# ── Project 2 starter: corpus survey. Pick one and go. ───────────────────────────
print(f'{"text":<32}{"chars":>10}{"words":>10}{"vocab":>9}{"chars/word":>12}')
print('-' * 74)
for fid in gutenberg.fileids():
    raw = gutenberg.raw(fid)
    words = gutenberg.words(fid)
    print(f'{fid:<32}{len(raw):>10,}{len(words):>10,}{len(set(w.lower() for w in words)):>9,}'
          f'{len(raw)/len(words):>12.2f}')

print()
print('Guidance:')
print('  - under ~200k characters: use character level, or you will overfit a word')
print('    vocabulary almost immediately')
print('  - over ~1M characters: word level becomes viable and trains much faster,')
print('    since each token carries more information')
print('  - concatenating several texts by one author is a legitimate way to get more')
print('    data, but check that the styles are actually consistent')

# YOUR CODE: choose a corpus, build the GPT, train, sample, evaluate.

text                                 chars     words    vocab  chars/word
--------------------------------------------------------------------------
austen-emma.txt                    887,071   192,427    7,344        4.61
austen-persuasion.txt              466,292    98,171    5,835        4.75
austen-sense.txt                   673,022   141,576    6,403        4.75


bible-kjv.txt                    4,332,554 1,010,654   12,767        4.29
blake-poems.txt                     38,153     8,354    1,535        4.57
bryant-stories.txt                 249,439    55,563    3,940        4.49
burgess-busterbrown.txt             84,663    18,963    1,559        4.46
carroll-alice.txt                  144,395    34,110    2,636        4.23
chesterton-ball.txt                457,450    96,996    8,335        4.72
chesterton-brown.txt               406,629    86,063    7,794        4.72
chesterton-thursday.txt            320,525    69,213    6,349        4.63


edgeworth-parents.txt              935,158   210,663    8,447        4.44
melville-moby_dick.txt           1,242,990   260,819   17,231        4.77
milton-paradise.txt                468,220    96,825    9,021        4.84
shakespeare-caesar.txt             112,310    25,833    3,032        4.35
shakespeare-hamlet.txt             162,881    37,360    4,716        4.36
shakespeare-macbeth.txt            100,351    23,140    3,464        4.34


whitman-leaves.txt                 711,215   154,883   12,452        4.59

Guidance:
  - under ~200k characters: use character level, or you will overfit a word
    vocabulary almost immediately
  - over ~1M characters: word level becomes viable and trains much faster,
    since each token carries more information
  - concatenating several texts by one author is a legitimate way to get more
    data, but check that the styles are actually consistent


---
# Project 3 — A Domain-Adapted Assistant

**Builds on:** lessons 10, 11
**Difficulty:** harder · **Time:** 4–6 hours

Train one base model, then produce several specialised variants using LoRA
adapters — and demonstrate that swapping adapters swaps behaviour while the base
weights never move.

This is how adapter-based deployment actually works in production, and building it
once makes the whole approach concrete.

### Minimum viable version

1. Train a base GPT on a broad corpus (several Gutenberg texts concatenated).
2. Implement `LoRALinear` (lesson 11) and a function to apply it to a model.
3. Fine-tune **three** separate adapters on three distinct domains — Shakespeare,
   Austen, and the KJV Bible, say — from the same frozen base.
4. Show that each adapter improves perplexity on its own domain.
5. Save adapters separately from the base and demonstrate hot-swapping: load base
   once, attach adapter A, generate, detach, attach adapter B, generate.

**Bar to clear:** each adapter beats the base model on its own domain, adapters
total under 5% of base parameters, and generated samples are distinguishable by
domain in a blind read.

### Extensions

- **A rank sweep** per domain. Do harder domains need higher rank? Form a
  hypothesis first, then test it.
- **QLoRA**: quantize the base to 4 bits and repeat. What does it cost?
- **Adapter merging.** Fold `B@A` into the base weights (`merged_weight()`) and
  verify the outputs are identical. Then try averaging two adapters — what does the
  result behave like?
- **Catastrophic forgetting, measured.** Track each adapter's perplexity on the
  *other* domains. Plot the full matrix.
- **Which layers matter?** Adapt only early layers, only late layers, only the MLP.
  Where does the domain knowledge actually live?

### The interesting question

An adapter is a few hundred kilobytes against a multi-megabyte base. If a full
fine-tune stores an entire model per task and LoRA stores a rounding error, what
exactly is being stored in those few hundred kilobytes — and why is it enough?

In [6]:
# ── Project 3 starter: three domains from one corpus family ─────────────────────
DOMAINS = {
    'shakespeare': ['shakespeare-hamlet.txt', 'shakespeare-macbeth.txt',
                    'shakespeare-caesar.txt'],
    'austen':      ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt'],
    'bible':       ['bible-kjv.txt'],
}

texts = {name: ''.join(gutenberg.raw(f) for f in files) for name, files in DOMAINS.items()}

# A shared character alphabet across all domains, so one base model covers them all.
all_chars = sorted(set(''.join(texts.values())))
print(f'shared alphabet: {len(all_chars)} symbols')
print()
for name, t in texts.items():
    print(f'  {name:<12} {len(t):>10,} characters')

print()
print('Suggested split: train the base on a mix of all three (or on a fourth,')
print('unrelated text), then adapt to each domain separately. Training the base on')
print('the same data you adapt to would make the adapters look better than they are.')
print()
print('A neutral base corpus, well outside all three domains:')
neutral = 'melville-moby_dick.txt'
print(f'  {neutral}: {len(gutenberg.raw(neutral)):,} characters')

# YOUR CODE: base model, LoRALinear, three adapters, swapping, forgetting matrix.

shared alphabet: 83 symbols

  shakespeare     375,542 characters
  austen        2,026,385 characters
  bible         4,332,554 characters

Suggested split: train the base on a mix of all three (or on a fourth,
unrelated text), then adapt to each domain separately. Training the base on
the same data you adapt to would make the adapters look better than they are.

A neutral base corpus, well outside all three domains:
  melville-moby_dick.txt: 1,242,990 characters


---
# Project 4 — An Architecture Bake-Off

**Builds on:** lessons 03, 04, 05, 06, 07
**Difficulty:** moderate, but demanding about rigour · **Time:** 4–6 hours

Compare CNN, LSTM, and transformer on one task, under conditions fair enough that
the result means something. Most published comparisons are confounded; yours does
not have to be.

### Minimum viable version

1. Fix a task: sentiment classification on `movie_reviews`.
2. Fix everything shared: the split, the vocabulary, the sequence length, the
   optimizer, the schedule, the number of epochs.
3. Implement all three architectures **at matched parameter count** — within about
   10% of each other. This is the part that takes real effort, and it is the part
   that makes the comparison honest.
4. Train each with **three random seeds**.
5. Report mean and standard deviation of test accuracy, plus training time per epoch
   and peak memory.

**Bar to clear:** a results table with error bars, and a written conclusion that
distinguishes what your data supports from what you merely suspect.

### Extensions

- **Learning-curve study.** Retrain each at 10%, 25%, 50%, and 100% of the training
  data. The ranking will probably change with dataset size — that is the most
  useful finding available from this project.
- **Sequence-length sensitivity.** Truncate to 50, 100, 200, 400 tokens. Which
  architecture degrades most, and does the explanation match the theory?
- **Ablations.** Transformer without positional encoding. LSTM without gates (i.e.
  a plain RNN). CNN with a single kernel width. Each isolates one claim from the
  book.
- **Inference cost.** Latency per example and throughput at batch 1 versus batch 64.
  Accuracy is rarely the only thing that matters in deployment.

### The rigour that makes this worth doing

A single seed tells you almost nothing: on 2,000 reviews, seed variance alone is
often ±2% accuracy. If architecture A beats B by 1% on one run, you have measured
noise and reported it as a finding. Three seeds is the minimum that lets you say
anything, and reporting the standard deviation is what separates an experiment from
an anecdote.

In [7]:
# ── Project 4 starter: fixed splits, shared vocabulary, a parameter-matching tool ─
docs = [(list(movie_reviews.words(f)), movie_reviews.categories(f)[0])
        for f in movie_reviews.fileids()]
random.Random(SEED).shuffle(docs)

cut = int(0.8 * len(docs))
train_docs, test_docs = docs[:cut], docs[cut:]

SEQ_LEN = 200
vocab = Vocab([[w.lower() for w in d] for d, _ in train_docs], max_size=10000)

def encode_docs(ds):
    X = torch.tensor([vocab.pad_to(vocab.encode([w.lower() for w in d], SEQ_LEN,
                                                add_special=False)[:SEQ_LEN], SEQ_LEN)
                      for d, _ in ds], dtype=torch.long)
    y = torch.tensor([1 if lab == 'pos' else 0 for _, lab in ds], dtype=torch.long)
    return X, y

X_tr, y_tr = encode_docs(train_docs)
X_te, y_te = encode_docs(test_docs)

print(f'train {tuple(X_tr.shape)}   test {tuple(X_te.shape)}   vocab {len(vocab):,}')
print(f'class balance (train): {y_tr.float().mean():.3f} positive')
print()


def match_params(build_fn, target, lo=8, hi=512, tol=0.10):
    """Binary-search a width so that build_fn(width) lands within `tol` of `target`.

    Use this to put every architecture on the same parameter budget — otherwise you
    are comparing capacity, not architecture.
    """
    best = None
    while lo <= hi:
        mid = (lo + hi) // 2
        n = sum(p.numel() for p in set(build_fn(mid).parameters()))
        if best is None or abs(n - target) < abs(best[1] - target):
            best = (mid, n)
        if n < target:
            lo = mid + 1
        else:
            hi = mid - 1
    width, n = best
    ok = abs(n - target) / target <= tol
    print(f'  width={width} -> {n:,} params  (target {target:,}, '
          f'off by {100*abs(n-target)/target:.1f}%) {"OK" if ok else "ADJUST"}')
    return width


# Example: an embedding-only baseline to set the budget everything else must match.
class BagOfEmbeddings(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), d, padding_idx=PAD_IDX)
        self.fc = nn.Linear(d, 2)

    def forward(self, x):
        mask = (x != PAD_IDX).unsqueeze(-1).float()
        pooled = (self.emb(x) * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(pooled)


baseline = BagOfEmbeddings(64).to(device)
TARGET = sum(p.numel() for p in set(baseline.parameters()))
print(f'parameter budget set by the bag-of-embeddings baseline: {TARGET:,}')
print()
print('Now match your CNN, LSTM, and transformer to it:')
print('  width = match_params(lambda d: MyCNN(d), TARGET)')

# YOUR CODE: three architectures, three seeds each, a results table with error bars.

train (1600, 200)   test (400, 200)   vocab 10,000
class balance (train): 0.491 positive

parameter budget set by the bag-of-embeddings baseline: 640,130

Now match your CNN, LSTM, and transformer to it:
  width = match_params(lambda d: MyCNN(d), TARGET)


---
## Before You Call Any Of These Done

A checklist. Most of these are failures that look like successes, which is why they
are worth checking deliberately rather than trusting that you would have noticed.

**Data**
- Vocabulary and any normalisation statistics built from the **training split only**
- Train and test sets genuinely disjoint (check for duplicate documents)
- Ten examples printed and read by eye — decoded back from the tensors, not from
  the source text
- `<UNK>` rate measured and reported

**Model**
- Initial loss ≈ `ln(n_classes)` or `ln(vocab_size)`
- Eight examples overfitted to near-zero loss before the full run
- Parameter count printed
- Shapes annotated at every non-obvious line

**Training**
- `model.train()` / `model.eval()` in the right places
- `optimizer.zero_grad()` present
- Gradient clipping on any sequence model
- Best checkpoint restored, not the last
- Learning rate printed for the first few epochs

**Evaluation**
- Test set touched **once**, at the very end (use validation for everything else)
- At least three seeds, with the standard deviation reported
- A baseline to compare against — majority class, or a bag-of-words model
- Outputs read by a human, not just scored

**Reporting**
- A statement of what you cannot conclude, alongside what you can
- Wall-clock time and peak memory, not only accuracy

---

## Where to Go After This

The curriculum ends here, but the material has obvious next steps:

- **Subword tokenisation** — implement BPE from scratch. It is the last major
  component of a modern language model that this curriculum does not build.
- **RLHF / DPO** — how a base language model becomes an assistant. Conceptually
  distinct from everything here, and the natural sequel to lesson 10.
- **Retrieval augmentation** — embed a corpus, retrieve by nearest neighbour, and
  condition generation on what you retrieved.
- **MLX** — Apple's own array framework, designed for unified memory. Reimplementing
  lesson 10's GPT in MLX is an excellent way to learn what PyTorch abstracts away.
- **Read the papers.** *Attention Is All You Need* (2017), *LoRA* (2021), and
  *QLoRA* (2023) are all readable now in a way they would not have been before
  lesson 00 — you have implemented most of what they describe.